# 7.24 Super-resolution

Super-resolution is not just enlarging an image; it is deciding which high-frequency detail is justified by the low-resolution evidence.

Super-resolution starts with a small image and asks for a larger one that preserves content while inventing plausible detail only where the evidence permits. Interpolation supplies the baseline, learned residuals add detail, and PSNR measures pixel fidelity while reminding us that sharpness and truth are not identical.

Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
np.set_printoptions(precision=3, suppress=True)

## The concept, built once

We first implement the tiny arithmetic in the lesson. The fidelity formula is

$$\operatorname{PSNR}=10\log_{10}\left(rac{L^2}{rac{1}{N}\sum_p(I_p-\hat I_p)^2}ight).$$

The checked examples are nearest value $1$, bilinear value $1.5$, bicubic value $2.5$, and PSNR $18.06$ dB.

In [ ]:
def nearest_upsample(img, scale):
    return np.repeat(np.repeat(img, scale, axis=0), scale, axis=1)


def bilinear_half(a, b):
    return 0.5 * a + 0.5 * b


def cubic_dot(values):
    weights = np.array([-0.0625, 0.5625, 0.5625, -0.0625])
    return float(np.dot(weights, values))


def psnr(target, prediction, peak):
    mse = np.mean((target - prediction) ** 2)
    return float(10.0 * np.log10((peak ** 2) / mse))


def box_blur(img):
    padded = np.pad(img, 1, mode="edge")
    out = np.zeros_like(img)
    for y in range(img.shape[0]):
        for x in range(img.shape[1]):
            out[y, x] = padded[y:y + 3, x:x + 3].mean()
    return out


def super_resolve(low, scale, detail_gain=0.35):
    base = nearest_upsample(low, scale)
    detail = base - box_blur(base)
    return np.clip(base + detail_gain * detail, 0.0, 1.0)


patch = np.array([[1, 2], [3, 4]], dtype=float)
nearest = nearest_upsample(patch, 2)
assert nearest[1, 1] == 1
assert bilinear_half(1, 2) == 1.5
assert cubic_dot(np.array([1, 2, 3, 4], dtype=float)) == 2.5
checked_psnr = psnr(np.array([1, 2, 3, 4], dtype=float), np.array([1, 1, 3, 4], dtype=float), 4)
assert round(checked_psnr, 2) == 18.06
print("nearest", nearest[1, 1])
print("bilinear", bilinear_half(1, 2))
print("bicubic", cubic_dot(np.array([1, 2, 3, 4], dtype=float)))
print("PSNR", round(checked_psnr, 2))

## Dataset ladder

For this geometry topic we build five low/high-resolution pairs, not the classification ladder. The rungs increase scale, texture, noise, and aliasing while keeping the same PSNR evaluation.

In [ ]:
def make_high_res(size, frequency, noise, seed):
    local_rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:size, 0:size]
    smooth = 0.45 + 0.25 * np.sin(xx / frequency) + 0.20 * np.cos(yy / (frequency + 1))
    edge = (xx > size * 0.45).astype(float) * 0.25
    checker = ((xx // max(1, int(frequency))) + (yy // max(1, int(frequency)))) % 2
    image = smooth + edge + 0.12 * checker
    image = image + local_rng.normal(0.0, noise, size=(size, size))
    return np.clip(image, 0.0, 1.0)


def downsample_mean(img, scale):
    h = img.shape[0] // scale
    w = img.shape[1] // scale
    blocks = img[:h * scale, :w * scale].reshape(h, scale, w, scale)
    return blocks.mean(axis=(1, 3))


def build_sr_ladder():
    specs = [
        ("D1 tiny edge", 16, 2, 5, 0.00),
        ("D2 shapes", 24, 2, 4, 0.01),
        ("D3 texture", 32, 2, 3, 0.02),
        ("D4 x4 scale", 32, 4, 3, 0.03),
        ("D5 noisy aliasing", 40, 4, 2, 0.06),
    ]
    rungs = []
    for seed, spec in enumerate(specs):
        name, size, scale, frequency, noise = spec
        high = make_high_res(size, frequency, noise, seed)
        low = downsample_mean(high, scale)
        rungs.append({"name": name, "high": high, "low": low, "scale": scale})
    return rungs


sr_rungs = build_sr_ladder()
for rung in sr_rungs:
    print(rung["name"], "low", rung["low"].shape, "high", rung["high"].shape, "scale", rung["scale"])

fig, axes = plt.subplots(1, 5, figsize=(12, 2.4))
for ax, rung in zip(axes, sr_rungs):
    ax.imshow(rung["low"], cmap="gray", vmin=0, vmax=1)
    ax.set_title(rung["name"])
    ax.axis("off")
plt.tight_layout()

## Run the same method across D1-D5

In [ ]:
sr_results = []
for rung in sr_rungs:
    pred = super_resolve(rung["low"], rung["scale"])
    score = psnr(rung["high"], pred, 1.0)
    sr_results.append({"name": rung["name"], "pred": pred, "psnr": score})

print("rung                 PSNR dB")
for result in sr_results:
    print(f"{result['name']:<20} {result['psnr']:.2f}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(13, 5))
for i, rung in enumerate(sr_rungs):
    axes[0, i].imshow(sr_results[i]["pred"], cmap="gray", vmin=0, vmax=1)
    axes[0, i].set_title(rung["name"])
    axes[0, i].axis("off")

scores = [item["psnr"] for item in sr_results]
axes[1, 0].plot(range(1, 6), scores, marker="o")
axes[1, 0].set_xlabel("rung")
axes[1, 0].set_ylabel("PSNR dB")
axes[1, 0].set_title("fidelity vs complexity")
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()

## Pitfall on D5: high PSNR is not the same as faithful evidence

A waxy average can score well, while a sharp hallucination may look crisp but violate downsample consistency.

In [ ]:
hard = sr_rungs[-1]
waxy = nearest_upsample(hard["low"], hard["scale"])
sharp = np.clip(super_resolve(hard["low"], hard["scale"], detail_gain=1.4), 0.0, 1.0)
waxy_psnr = psnr(hard["high"], waxy, 1.0)
sharp_psnr = psnr(hard["high"], sharp, 1.0)
waxy_consistency = np.mean((downsample_mean(waxy, hard["scale"]) - hard["low"]) ** 2)
sharp_consistency = np.mean((downsample_mean(sharp, hard["scale"]) - hard["low"]) ** 2)
print("waxy PSNR", round(waxy_psnr, 2), "consistency", round(waxy_consistency, 5))
print("sharp PSNR", round(sharp_psnr, 2), "consistency", round(sharp_consistency, 5))
print("fix: report downsample consistency beside PSNR")

## Evaluate it + Practice

- Metric: PSNR against the synthetic high-resolution target; no-skill baseline is nearest upsampling.
- Sanity check: D1 should reconstruct better than D5.
- Ablation: set `detail_gain=0` and the residual idea disappears.
- Failure signal: sharp detail with poor downsample consistency is hallucination risk.

Practice:
1. Change D5 noise and watch PSNR move.
2. Compare bilinear-style smoothing against nearest.
3. Add a residual gain sweep.